# Secondary cracking after a sudden fracture event — theory

Companion notebook to [`problems/secondary.py`](secondary.py), following the concept note (A.L.B., 1st June 2026).

## The question

> When a crack suddenly appears in a thin film glued to a substrate, can the sudden release of stress create more cracks?

The question is simple, but it holds the main ingredients of dynamic fracture: **stored elastic energy is released all at once**, **waves travel**, **they reflect at the boundaries**, **they are damped**, and **a cracking rule may switch on again** away from the first crack. We are not after a detailed simulation. We want a **short, clear rule** that says **when** a second crack can appear, **when** it cannot, and **which dimensionless numbers** control the switch.

## The physical story in four steps

The notebook follows one chain of cause and effect:

1. **A first crack appears.** Where the film was uniformly stretched, there is now a free (traction-free) face. The film next to it is suddenly out of balance.
2. **A stress-release pulse is sent out.** The film relaxes toward a new balance, but it overshoots: the stored energy turns into a travelling, *spreading*, *damped* wave.
3. **The pulse reflects and overlaps itself.** In the finite half-specimen the wave bounces between the crack and the far end and adds to itself — the simplest way to **refocus** stress.
4. **A second crack may form.** If, somewhere away from the first crack, the *dynamic* strain briefly passes the material's cracking threshold **before** damping kills the wave, a new crack is born.

The goal is to put a simple formula (or one line of numpy) on each step, so the final answer is an inequality between a few dimensionless numbers.

## Why this geometry

We use a **1D film on a linear elastic (Winkler) foundation**. A Winkler foundation treats the substrate as a row of independent springs of stiffness $k$ per unit length that pull the film back to its glued position. It is the cheapest model that still has everything we need:

- a real **elastic length** $\ell_e$ (the distance over which stress relaxes near a free edge),
- **spreading** waves (a substrate cut-off frequency),
- **damping**,
- and **stress refocusing** by reflection,

while still being **solvable by hand** — every step below comes down to elementary functions and one Fourier-cosine sum.

> **⚠️ Notation warning — read this first.**
> Several symbols in the concept note clash with the repository-wide names in [`tools/parameters.py`](../tools/parameters.py). To avoid silent bugs, this problem keeps its **own** parameter dictionary, `SECONDARY_PARAMETERS`, **at the end of** [`problems/secondary.py`](secondary.py) — *not* in `tools/parameters.py`.
>
> | symbol | meaning in `tools/parameters.py` | meaning **in this note** |
> |---|---|---|
> | $\eta$ | dynamic *loading* time-scale ($\tau=\eta t$) | **viscous damping** coefficient of the PDE |
> | $\Lambda$ | foundation stiffness | $\ell_e/\ell$ — here renamed `Lambda_bar` |
> | $\gamma$ | Newmark time-integration coefficient | **modal damping rate** $\eta/(2m)$ |
> | $\ell$ / `l_hat` | phase-field regularisation length | **half-specimen length** (the regularisation length is $\ell_d$ here; the code's control parameter `l_hat` is the ratio $\ell_d/\ell$) |
>
> Throughout we set $\ell_e=c=\theta=E_h=m=1$, so only the **three** real control parameters are left: $\Lambda=\ell_e/\ell$, the damping number $\Gamma=\gamma\,\tau_{rt}$, and the damage-length ratio $\hat\ell=\ell_d/\ell$.

In [7]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Auto-reload edited modules (tools/, problems/) before each cell runs, so
# editing e.g. tools/plotting.py takes effect WITHOUT restarting the kernel.
%load_ext autoreload
%autoreload 2

ROOT = Path.cwd().resolve()
if ROOT.name == "problems":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from problems import secondary as sc
from tools.solvers import critical_strain

# IMPORTANT: set the inline backend AFTER the imports above. `problems.secondary`
# pulls in `tools/imports.py`, which calls matplotlib.use("Agg") (headless-safe)
# at import time -- doing this earlier would be silently overridden, leaving
# plt.show() on the non-interactive Agg backend (figures never display).
%matplotlib inline
plt.close("all")   # drop any half-rendered figures left over from a failed run

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Physical setting and the governing equation

### 1.1 The film, the substrate, and the pre-stress

Think of a thin elastic film glued onto a much stiffer substrate. We follow only the **axial** displacement $u(x,t)$ of the film (in-plane, along $x$). Three constants describe it:

| symbol | meaning | dimension |
|---|---|---|
| $E_h \equiv E_f h_f$ | **axial stiffness** (Young's modulus $\times$ thickness; $h$ is a label, not a factor) | force |
| $m$ | **mass per unit length** | mass/length |
| $k$ | **foundation stiffness** — restoring force per unit length, per unit displacement | force/length² |

The substrate acts as a **Winkler foundation**: every point of the film is tied to its glued position by a spring of stiffness $k$. So a film displacement $u$ costs a restoring force $-k\,u$ per unit length and stores energy density $\tfrac12 k u^2$. (Physically, $k$ comes from the substrate's shear stiffness resisting film/substrate sliding.)

**Loading.** Before any crack, the film carries an imposed mismatch (eigen-)strain $\theta$ (e.g. thermal: film and substrate have different natural lengths). With the film glued along its whole length, the only uniform solution is $u_{\rm before}\equiv 0$ (Section 2.2). The elastic strain is then $e_0=u_x-\theta=-\theta$, so the film sits under a uniform **pre-stress** $N_0=-E_h\theta$; the substrate springs store nothing because $u_{\rm before}=0$.

### 1.2 Strain, sign convention, and the reference state ⚠️

Getting the reference state right is the crux of the whole static analysis — a wrong choice quietly breaks the eigenfunction expansion later. We follow the **corrected concept note** (`secondary_cracking_corrected.pdf`, ALB). The elastic (stress-producing) strain is

$$ \boxed{\,e(x,t) = u_x(x,t) - \theta\,}\qquad N(x,t)=E_h\,e=E_h\,(u_x-\theta), $$

with $u$ measured from the substrate's relaxed configuration and $\theta$ the imposed mismatch eigenstrain.

- **Before the crack:** $u_{\rm before}\equiv 0$ (the *only* uniform solution — **not** $\theta x$) $\Rightarrow e_0=-\theta$, $N_0=-E_h\theta$ (a uniform pre-stress).
- A **traction-free** face means $N=0\Rightarrow e=0\Rightarrow u_x=+\theta$ there: the film end moves to shed its stress.

Because every energy below is quadratic in $\theta$ and the cracking test uses $|e|$, **none of the final numbers depend on the sign of this convention** (the PDF makes the same remark). Fixing it the PDF's way has two payoffs: the formulas read line-for-line against the document, and — crucially — the initial data stay consistent with the boundary conditions of the modal problem (Section 3), which a careless "$u_{\rm before}=\theta x$" ansatz would violate (Section 2.3). The repository code now uses exactly this convention (see [`relaxed_strain`](secondary.py)/[`strain_field`](secondary.py), which build $e$ as `u_x - theta`).

### 1.3 The post-crack dynamic equation

Once the crack opens at $x=0$, Newton's law per unit length of film is (inertia $+$ drag $=$ internal-force gradient $-$ foundation pull):

$$ \underbrace{m\,u_{tt}}_{\text{inertia}} + \underbrace{\eta\,u_t}_{\text{viscous damping}} = \underbrace{\partial_x N}_{\;=\,E_h u_{xx}} \;-\; \underbrace{k\,u}_{\text{foundation}} ,$$

that is

$$ \boxed{\,m\,u_{tt} + \eta\,u_t - E_h\,u_{xx} + k\,u = 0\,.} $$

Here $\eta\ge 0$ is a **viscous damping coefficient** (a drag on the film's velocity). This is a *damped Klein–Gordon* / telegraph equation — a wave equation with the extra term $ku$, and that extra term is exactly what makes the substrate physics interesting.

### 1.4 Derived quantities

From $(E_h,m,k)$ and the damping $\eta$ we get four more quantities:

$$ c=\sqrt{\tfrac{E_h}{m}},\qquad \ell_e=\sqrt{\tfrac{E_h}{k}},\qquad \omega_0=\sqrt{\tfrac{k}{m}}=\tfrac{c}{\ell_e},\qquad \gamma=\tfrac{\eta}{2m}, $$

with the handy identity $k=E_h/\ell_e^2$.

- **$c$ — wave speed** of the *free* film (no substrate). The usual $\sqrt{\text{stiffness}/\text{inertia}}$.
- **$\ell_e$ — elastic transfer length.** How far the substrate lets film stress relax near a free edge. It is the key length here: a crack only "feels" the film over a region $\sim\ell_e$ around it. (Set $u_{tt}=u_t=0$: $-E_h u_{xx}+ku=0$ has solutions $e^{\pm x/\ell_e}$, hence the name.)
- **$\omega_0$ — substrate cut-off frequency.** No wave below $\omega_0$ can travel (Section 3). It is set by the foundation-to-inertia ratio.
- **$\gamma$ — modal damping rate.** Each mode decays like $e^{-\gamma t}$. The $1/2$ is the usual factor that makes $\gamma$ the amplitude (not energy) decay rate.

### 1.5 The three control parameters

After setting $\ell_e=c=\theta=E_h=m=1$ (which forces $k=1$, $\omega_0=1$, and leaves $\eta=2\gamma$), the whole problem depends on just **three** numbers:

$$ \Lambda=\frac{\ell_e}{\ell}\ \ \text{(geometry: transfer vs. specimen length)},\qquad \Gamma=\gamma\,\tau_{rt}\ \ \text{(damping per wave round-trip)},\qquad \hat\ell=\frac{\ell_d}{\ell}\ \ \text{(damage vs. specimen length)}, $$

with $\tau_{rt}$ the round-trip time of the pulse (Section 4). Note the flip from an earlier draft: **$\Lambda$ is now $\ell_e/\ell$** (small $\Lambda$ = long film, so only a boundary layer of size $\ell_e$ relaxes; large $\Lambda$ = short film). In the code these are `Lambda_bar`, `Gamma` and `l_hat` in `SECONDARY_PARAMETERS`. The third one enters only through the **nucleation threshold**: with $\ell_d=\hat\ell\,\ell$ the criterion of Section 5 gives $e_{crit}\sim1/\sqrt{\ell_d}$, so $\hat\ell$ (together with $\Lambda$, since $\ell=\ell_e/\Lambda$) sets the ratio $e_{crit}/\theta$ the film must overshoot. Everything below is, in the end, a function of $(\Lambda,\Gamma,\hat\ell)$.

## 2. Static release: how much energy does the first crack liberate?

Before the *dynamics*, we ask the static question: **once everything has settled**, what new balance does the cracked half-specimen reach, and how much stored energy was let go? This sets the energy budget for waves, damping, and second cracks.

### 2.1 The boundary-value problem

By symmetry we look at **one half-specimen** $x\in[0,\ell]$, with the crack at $x=0$ and the far (symmetry) end at $x=\ell$. The relaxed balance $u^+(x)$ is the static solution ($u_{tt}=u_t=0$):

$$ -E_h\,u^+_{xx} + k\,u^+ = 0 \;\;\Longleftrightarrow\;\; u^+_{xx} = \frac{u^+}{\ell_e^2}, $$

a linear ODE with general solution $u^+ = P\cosh\!\frac{x}{\ell_e}+Q\sinh\!\frac{x}{\ell_e}$, or the same in terms of $\ell-x$.

**Two boundary conditions** (with $e=u_x-\theta$ from §1.2):

1. **Traction-free crack face** at $x=0$: $\;N(0)=0\Rightarrow e^+(0)=0\Rightarrow u^+_x(0)=+\theta$.
2. **Match at the far end** $x=\ell$: the film is still glued and recovers the pre-crack state, $\;u^+(\ell)=0$.

Writing $u^+=A\,\sinh\!\frac{\ell-x}{\ell_e}$ satisfies BC 2 automatically. Then $u^+_x=-\tfrac{A}{\ell_e}\cosh\!\frac{\ell-x}{\ell_e}$, and BC 1 gives $-\tfrac{A}{\ell_e}\cosh\!\frac{\ell}{\ell_e}=+\theta$, i.e. $A=-\theta\ell_e/\cosh\!\frac{\ell}{\ell_e}$. So (PDF eqs 6–7)

$$ \boxed{\,u^+(x) = -\,\theta\,\ell_e\,\frac{\sinh\!\big(\tfrac{\ell-x}{\ell_e}\big)}{\cosh\!\big(\tfrac{\ell}{\ell_e}\big)}\,},\qquad u^+_x(x)=\theta\,\frac{\cosh\!\big(\tfrac{\ell-x}{\ell_e}\big)}{\cosh\!\big(\tfrac{\ell}{\ell_e}\big)},\qquad \boxed{\,e^+(x) = u^+_x-\theta = \theta\Big(\frac{\cosh\!\big(\tfrac{\ell-x}{\ell_e}\big)}{\cosh\!\big(\tfrac{\ell}{\ell_e}\big)}-1\Big)\,}. $$

These are exactly [`relaxed_displacement`](secondary.py) and [`relaxed_strain`](secondary.py). Two checks: $u^+_x(0)=\theta$ so $e^+(0)=0$ (all stress released at the crack, $N=0$ ✓), and far from the crack $e^+\to-\theta$ over a few $\ell_e$ (it recovers the pre-stress ✓). **The relaxation lives in a thin layer of width $\ell_e$** — the key geometric fact of the whole problem. (Numerically: at $\ell/\ell_e=3$, i.e. $\Lambda=1/3$, the code gives $A_0=0.52322$ and $u^+(0)=-\theta\ell_e\tanh(\ell/\ell_e)$, matching the PDF.)

> **Why a naive ansatz fails.** If one instead takes the pre-crack state as $u_{\rm before}=\theta x$ and the far-field as $u^+(\ell)=\theta\ell$, an extra term $\theta x$ contaminates $u^+$ and gives $u^+_x(0)=2\theta\neq\theta$ — the traction-free condition is violated. The fix is **not** to patch the sign of the hyperbolic term but to use the correct reference state $u_{\rm before}=0$ (§1.2) and far-field $u^+(\ell)=0$. This is exactly what keeps the modal initial/boundary data consistent in Section 3.

### 2.2 The released energy

The stored potential energy on the half-specimen is elastic $+$ foundation:

$$ \mathcal E[u] = \int_0^\ell \Big(\tfrac12 E_h\,e^2 + \tfrac12 k\,u^2\Big)\,dx,\qquad e=u_x-\theta. $$

- **Before** the crack: $u\equiv0$, $e_0=-\theta$, springs relaxed $\Rightarrow \mathcal E_{\rm before}=\tfrac12 E_h\theta^2\ell$.
- **After** relaxing: put in $u^+,e^+$. Integrating by parts (using $u^+_{xx}=u^+/\ell_e^2$, $k=E_h/\ell_e^2$) collapses everything to a single $\tanh$ (PDF eq 9):

$$ \boxed{\;\Delta E_{1/2}=\mathcal E_{\rm before}-\mathcal E_{\rm after}= \tfrac12\,E_h\,\theta^2\,\ell_e\,\tanh\!\Big(\frac{\ell}{\ell_e}\Big)= \tfrac12\,E_h\,\theta^2\,\ell_e\,\tanh\!\Big(\frac{1}{\Lambda}\Big)\;} $$

returned by [`released_energy`](secondary.py). (A central crack in a *full* specimen relaxes two faces, so the note's total is twice this, $\Delta E_{\rm tot}=E_h\theta^2\ell_e\tanh(\ell/\ell_e)$.)

### 2.3 The two regimes — read off the geometry

$$ \Delta E_{1/2}\;\sim\;\tfrac12 E_h\theta^2\,\ell \;\;(\ell\ll\ell_e,\ \Lambda\gg1,\ \tanh(1/\Lambda)\approx1/\Lambda), \qquad \Delta E_{1/2}\;\sim\;\tfrac12 E_h\theta^2\,\ell_e \;\;(\ell\gg\ell_e,\ \Lambda\ll1,\ \tanh(1/\Lambda)\approx1). $$

- **Short film** $\ell\ll\ell_e$ (large $\Lambda$): the crack relaxes the **whole** film — released energy grows with $\ell$.
- **Long film** $\ell\gg\ell_e$ (small $\Lambda$): the crack relaxes only a **thin layer** of size $\ell_e$ — released energy **levels off** at $\tfrac12 E_h\theta^2\ell_e$, no matter how long the specimen is. *A long film does not give up all its stored energy when it cracks.*

This levelling-off is the first energy limit: only a finite budget $\sim\tfrac12 E_h\theta^2\ell_e$ is available to drive waves and maybe second cracks.

**The code cell below** checks the $\tanh$ formula against a direct trapezoidal integral of $\mathcal E_{\rm before}-\mathcal E_{\rm after}$ at several $\Lambda$ (note $k=E_h/\ell_e^2$, so the foundation density is $\tfrac12(E_h/\ell_e^2)u^2$, and $e^+,u^+$ enter only as squares — so the result is the same in either sign convention), and plots $\tanh(1/\Lambda)$ between the two limits.

In [ ]:
# Verify the released-energy formula against a direct quadrature.
# np.trapz was renamed np.trapezoid in NumPy 2.0; fall back so this runs on either.
trapz = getattr(np, "trapezoid", getattr(np, "trapz"))

theta, ell_e, E_h = 1.0, 1.0, 1.0
for Lam in (5.0, 1.0, 0.2, 0.05):          # Lambda = ell_e/ell
    ell = ell_e / Lam                       # long film <-> small Lambda
    x = np.linspace(0, ell, 20001)
    e_plus = sc.relaxed_strain(x, theta, ell, ell_e)
    u_plus = sc.relaxed_displacement(x, theta, ell, ell_e)
    E_before = 0.5 * E_h * theta**2 * ell                                  # uniform pre-crack strain theta
    E_after  = trapz(0.5 * E_h * e_plus**2 + 0.5 * (E_h / ell_e**2) * u_plus**2, x)  # elastic + foundation (k = E_h/ell_e^2)
    dE_formula = sc.released_energy(theta, ell, ell_e, E_h)
    print(f"Lambda={Lam:5.2f} (ell/ell_e={1/Lam:6.2f}):  dE(quadrature)={E_before - E_after:.6f}"
          f"   dE(tanh formula)={dE_formula:.6f}")

Lam_axis = np.linspace(0.1, 10, 200)
plt.figure(figsize=(5.5, 3.5))
plt.plot(Lam_axis, np.tanh(1.0 / Lam_axis), 'k-', label=r'$\tanh(1/\Lambda)$')
plt.plot(Lam_axis, np.clip(1.0 / Lam_axis, None, 1.4), 'b:', label=r'$1/\Lambda$ ($\ell\ll\ell_e$)')
plt.axhline(1, color='g', ls=':', label=r'$1$ ($\ell\gg\ell_e$)')
plt.xlabel(r'$\Lambda=\ell_e/\ell$'); plt.ylabel(r'$\Delta E_{1/2}/(\frac{1}{2} E_h\theta^2\ell_e)$')
plt.legend(); plt.grid(alpha=0.3)
plt.title(r'Step 1 static release: $\Delta E_{1/2}/(\frac{1}{2} E_h\theta^2\ell_e)=\tanh(1/\Lambda)$ vs $\Lambda$')
plt.show()

## 3. The modal wave picture: which wavelengths carry the energy?

### 3.1 Split off the new equilibrium

The instant the crack opens, the film **has not moved yet**: it is still $u(x,0)=0$ (the pre-crack state), but the *boundary condition* has jumped to traction-free. So the film is out of balance with respect to its new balance $u^+(x)$ from Section 2. Split the motion into the new balance plus a transient:

$$ v(x,t) = u(x,t) - u^+(x). $$

Since $u^+$ is itself a static solution, $v$ obeys the **same homogeneous PDE** $\,m v_{tt}+\eta v_t-E_h v_{xx}+kv=0$, but now with **homogeneous boundary conditions** and a non-zero start:

$$ v_x(0,t)=0,\qquad v(\ell,t)=0,\qquad v(x,0)=-u^+(x),\qquad v_t(x,0)=0. $$

The start follows from $u(x,0)=0$; the velocity is zero because nothing has moved yet. The boundary conditions are the *differences* between the $u$-conditions and the $u^+$-conditions, which is why they become homogeneous: for $t>0$ the free-face condition is $u_x(0,t)=+\theta$ and $u^+_x(0)=+\theta$, so $v_x(0,t)=0$ (PDF eq 11). This is exactly the start set in [`modal_coefficients`](secondary.py) (`v0 = -relaxed_displacement(...)`).

### 3.2 Eigenfunctions and the dispersion relation

The homogeneous conditions (Neumann at $x=0$, Dirichlet at $x=\ell$) pick the cosine basis

$$ \phi_n(x)=\cos(q_n x),\qquad q_n=\frac{(n+\tfrac12)\pi}{\ell},\qquad n=0,1,2,\dots $$

(Check: $\phi_n'(0)=0$ ✓; $\phi_n(\ell)=\cos((n+\tfrac12)\pi)=0$ ✓.) Putting one mode $v=\cos(q_n x)\,e^{i\omega t}$ into the *undamped* PDE ($-m\omega^2+E_h q_n^2+k=0$, divided by $m$) gives the **Winkler dispersion relation**

$$ \boxed{\,\omega_n^2 = \omega_0^2 + c^2 q_n^2\,}\qquad\Big(\text{continuum form } \omega^2=\omega_0^2+c^2q^2\Big), $$

built in [`modal_setup`](secondary.py). **Three consequences:**

1. **Cut-off frequency.** $\omega\ge\omega_0$ always: the substrate blocks any oscillation slower than $\omega_0=c/\ell_e$. (A free bar, $k=0$, has no cut-off.)
2. **Dispersion.** $\omega$ is *not* proportional to $q$, so different wavelengths move at different speeds — a localised pulse spreads out as it travels.
3. **Group velocity.** Energy travels at $v_g=\dfrac{d\omega}{dq}=\dfrac{c^2 q}{\omega}$, which is $<c$ and depends on $q$. Long waves ($q\to0$) hardly move ($v_g\to0$); short waves ($q\to\infty$) approach the free-bar speed $c$.

### 3.3 The modal spectrum is *exactly* a Lorentzian filter

Project the starting mismatch onto the basis (the modes are orthogonal with $\int_0^\ell\cos^2 q_n x\,dx=\ell/2$). Writing $v(x,0)=-u^+(x)=\theta\ell_e\sinh\big((\ell-x)/\ell_e\big)/\cosh(\ell/\ell_e)$,

$$ A_n=a_n(0)=\frac{2}{\ell}\int_0^\ell v(x,0)\,\cos(q_n x)\,dx. $$

This integral has a closed form: integrate by parts twice and use $u^+_{xx}=u^+/\ell_e^2$ (the ODE $u^+$ obeys), which makes the integral reproduce itself so you can solve for it. The result is **not approximate — it is exact** (PDF eq 16):

$$ \boxed{\,a_n(0) = \frac{2\theta\,\ell_e^2/\ell}{\,1+(q_n\ell_e)^2\,}\,}=\frac{2\theta\,\ell_e^2/\ell}{1+\beta_n^2},\qquad \beta_n\equiv q_n\ell_e. $$

This is a **pure positive Lorentzian** in $\beta_n$ — *no* $(-1)^n$ factor — exactly the note's filter $A_n\sim 1/(1+(q_n\ell_e)^2)$ (the code's numerical projection matches it to $\sim10^{-7}$; e.g. $A_0=0.52322$ at $\ell/\ell_e=3$, i.e. $\Lambda=1/3$, the PDF's quoted value). What it means: the first crack does **not** excite all modes equally — it is a low-pass filter with corner at $\beta_n\sim1$. It mostly launches wavelengths $\gtrsim\ell_e$ and strongly suppresses short ones $\beta_n\gg1$ (where $a_n\sim 1/q_n^2$).

> **Why only $1/q_n^2$ decay? Because the crack is a "shock".** The starting data $-u^+(x)$ does **not** match the Neumann condition at the crack: $\partial_x[-u^+]_{x=0}=-\theta\neq0$, while every basis function has zero slope there. This *mismatch between the starting data and the new boundary condition* is exactly what makes the event a stress-release **shock**: a cosine series whose terms all have zero edge-slope can only build a function with non-zero edge-slope by adding up infinitely many modes, and then the spectrum decays slowly ($\sim1/q^2$). In short, suddenly making a free edge injects a *broadband* pulse — just the ingredient needed to (maybe) re-crack the film elsewhere.

### 3.4 The modal energy quota

Each mode carries energy $\propto \omega_n^2 a_n^2$ (kinetic + potential of a unit-mass oscillator of frequency $\omega_n$, amplitude $a_n$). The **normalised energy quota**

$$ Q_n=\frac{\omega_n^2\,a_n^2}{\sum_j \omega_j^2\,a_j^2}=\frac{(1+\beta_n^2)^{-1}}{\sum_j(1+\beta_j^2)^{-1}} $$

(in [`modal_energy_quota`](secondary.py)), plotted against $\beta_n=q_n\ell_e$, shows *which wavelengths carry the released energy*. The $\omega_n^2$ weight partly offsets the $a_n^2\sim1/\beta_n^4$ suppression, so the energy peaks at middling $\beta_n\lesssim1$ — wavelengths near the transfer length $\ell_e$. This answers note questions 2–3 (*which wavelengths carry the energy, and how $\ell_e$ filters the spectrum*).

**In the code cell below:** the markers are the amplitudes/quota from [`modal_coefficients`](secondary.py) (numerical projection), the dashed red line is the exact Lorentzian $1/(1+\beta_n^2)$. The markers are normalised by $|a_0|$, so they trace $(1+\beta_0^2)/(1+\beta_n^2)$ — the same shape, shifted only by the small constant $1+\beta_0^2$; §3.3 shows the *un-normalised* match is exact.

In [ ]:
Lam = 0.2                    # Lambda = ell_e/ell  ->  ell = ell_e/Lam = 5 (a long film)
ell = ell_e / Lam
q, omega = sc.modal_setup(ell, ell_e, c=1.0, n_modes=200)
a0 = sc.modal_coefficients(theta, ell, ell_e, q)
Q = sc.modal_energy_quota(omega, a0)
beta = q * ell_e

# Exact closed form of section 3.3:  a_n(0) = -(2 theta ell_e^2/ell) / (1+beta_n^2).
# Compare the *un-normalised* amplitude so the agreement is exact, not just same-shape.
a0_scaled = np.abs(a0) * ell / (2 * theta * ell_e**2)      # -> 1/(1+beta^2)
lorentz   = 1.0 / (1.0 + beta**2)
print("max |numeric - exact Lorentzian| over all modes:", np.max(np.abs(a0_scaled - lorentz)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.loglog(beta, a0_scaled, 'ko', ms=4, label=r'$|a_n(0)|\,\ell/(2\theta\ell_e^2)$ (computed)')
ax1.loglog(beta, lorentz, 'r--', lw=2, label=r'exact $1/(1+\beta_n^2)$')
ax1.set_xlabel(r'$\beta_n=q_n\ell_e$'); ax1.set_ylabel(r'$|a_n(0)|\,\ell/(2\theta\ell_e^2)$')
ax1.legend(); ax1.grid(alpha=0.3, which='both')
ax1.set_title(r'Modal filtering: $|a_n(0)|\,\ell/(2\theta\ell_e^2)=1/(1+\beta_n^2)$ vs $\beta_n$')
ax2.semilogx(beta, Q, 'ko-', ms=3)
ax2.axvline(1, color='r', ls='--', label=r'$\beta_n=1$  (energy peaks near $\ell_e$)')
ax2.set_xlabel(r'$\beta_n=q_n\ell_e$'); ax2.set_ylabel(r'$Q_n=\omega_n^2 a_n^2/\sum_j\omega_j^2 a_j^2$')
ax2.legend(); ax2.grid(alpha=0.3, which='both')
ax2.set_title(r'Modal energy quota: $Q_n$ vs $\beta_n$')
plt.tight_layout(); plt.show()

## 4. Reflection, superposition, and damping

### 4.1 The round-trip time of the released packet

Most of the released energy is in modes with $\beta_n=q_n\ell_e\sim\mathcal O(1)$ (Section 3.4), so take the **representative component** $q\sim 1/\ell_e$ as a stand-in for the pulse. Its frequency and group velocity come from the dispersion relation (recall $\omega_0=c/\ell_e$):

$$ \omega=\sqrt{\omega_0^2+c^2q^2}\;\Big|_{q=1/\ell_e}=\sqrt{2}\,\omega_0,\qquad v_g=\frac{c^2 q}{\omega}\Big|_{q=1/\ell_e}=\frac{c^2/\ell_e}{\sqrt2\,c/\ell_e}=\frac{c}{\sqrt2}. $$

The pulse leaves the crack ($x=0$), travels to the far end ($x=\ell$), reflects, and comes back — a distance $2\ell$ at speed $v_g$. So the **round-trip time** is

$$ \boxed{\,\tau_{rt}\sim\frac{2\ell}{v_g}\sim 2\sqrt2\,\frac{\ell}{c}\,}, $$

exactly the `tau_rt = 2*sqrt(2)*ell/c` in [`run_problem`](secondary.py). This is the natural clock of the problem: a second crack, if it happens, happens *because the wave returns and overlaps itself*, so we measure damping against one round trip.

### 4.2 The damping number $\Gamma$

Damping makes each mode decay like $e^{-\gamma t}$ (Section 4.3), so over one round trip the amplitude drops by $e^{-\gamma\tau_{rt}}=e^{-\Gamma}$. The single **damping number**

$$ \boxed{\,\Gamma=\gamma\,\tau_{rt}\,} $$

decides the fate of the pulse (in the code, `gamma_bar = Gamma / tau_rt` recovers $\gamma$ from a chosen $\Gamma$):

| regime | meaning |
|---|---|
| $\Gamma\ll1$ | waves survive many reflections — they **refocus** and may re-crack the film |
| $\Gamma\sim1$ | the pulse is noticeably weakened within one round trip |
| $\Gamma\gg1$ | the wave dies **before** it returns — no second crack |

### 4.3 Each mode is a damped oscillator

Put $v=\sum_n a_n(t)\cos(q_n x)$ into the full PDE. Orthogonality of the $\cos(q_n x)$ splits the modes into independent damped oscillators:

$$ \ddot a_n + 2\gamma\,\dot a_n + \omega_n^2\,a_n = 0 $$

(divide $m v_{tt}+\eta v_t-E_h v_{xx}+kv=0$ by $m$, use $v_{xx}\to-q_n^2$, and $E_h/m=c^2$, $k/m=\omega_0^2$ so that $c^2 q_n^2+\omega_0^2=\omega_n^2$). With the **start from rest** $a_n(0)$ from §3.3 and $\dot a_n(0)=0$, the under-damped solution is (PDF eq 18)

$$ \boxed{\,a_n(t)=a_n(0)\,e^{-\gamma t}\Big(\cos\Omega_n t+\frac{\gamma}{\Omega_n}\sin\Omega_n t\Big)\,},\qquad \Omega_n=\sqrt{\omega_n^2-\gamma^2}, $$

in [`modal_evolution`](secondary.py). The code takes $\Omega_n$ as a **complex** square root, so the *same* formula still works in the over-damped case $\gamma>\omega_n$ (then $\cos,\sin\to\cosh,\sinh$ and the mode decays without oscillating) — only the real part is kept. Each mode keeps its *own* frequency $\omega_n$ but shares the *same* decay rate $\gamma$: damping is the same across wavelengths.

### 4.4 Reconstructing the dynamic strain

Add up the modes to get the transient field, then differentiate for the strain. With $v_x=\sum_n a_n(t)\,\partial_x\cos(q_n x)=-\sum_n q_n a_n(t)\sin(q_n x)$ and $e=e^++v_x$ (§1.2, §3.1):

$$ v(x,t)=\sum_n a_n(t)\cos(q_n x),\qquad \boxed{\,e(x,t)=\underbrace{e^+(x)}_{\text{relaxed (static)}}\;-\;\underbrace{\sum_n q_n a_n(t)\sin(q_n x)}_{\text{travelling transient}}\,}, $$

exactly [`strain_field`](secondary.py) (PDF eq 20). **Check at $t=0$:** the $a_n(0)$ are the cosine coefficients of $-u^+$, so $v_x(x,0)=-u^+_x(x)$ and $e(x,0)=e^+(x)-u^+_x(x)=(u^+_x-\theta)-u^+_x=-\theta$ — the *uniform* pre-stress $e_0=-\theta$ everywhere, as it must be (nothing has moved yet, up to mode truncation). At the crack face $x=0$, $\sin(q_n\cdot 0)=0\Rightarrow v_x(0,t)=0$, so $e(0,t)=e^+(0)=0$ stays traction-free for all $t$. As $t$ grows the $\sin(q_n x)$ terms travel, reflect, and overlap; Section 5 asks whether their sum can briefly push $|e|$ past $e_{crit}$ somewhere away from the first crack.

Putting §4.1–§4.4 together: a second crack hinges on the competition between $\ell_e$, $\ell$, $c$, $\eta$ and the damage length $\ell_d$ — which collapse into just $(\Lambda,\Gamma,\hat\ell)$.

## 5. The damage criterion: when does a second crack nucleate?

Now we need a *material* statement: given the dynamic strain $e(x,t)$ from Section 4, when is the film locally damaged enough to break again? We reuse the **nucleation threshold of the gradient-damage (phase-field) model** used elsewhere in this repository (PDF Section 7).

### 5.1 The damage energy density

Add a damage variable $\alpha\in[0,1]$ ($0$ = intact, $1$ = fully broken). The 1D energy density is

$$ \psi(e,\alpha)=\underbrace{\tfrac12 E_h\,a(\alpha)\,e^2}_{\text{(degraded) elastic}} + \underbrace{\frac{G_c}{\ell_d}\,w(\alpha)}_{\text{local fracture}} + \underbrace{G_c\,\ell_d\,|\alpha_x|^2}_{\text{gradient}},\qquad e=u_x-\theta, $$

(the same $e=u_x-\theta$ as §1.2, so the strain that enters the criterion is exactly the dynamic strain of Section 4), with the standard sign assumptions

$$ a(0)=1,\quad a'(0)<0\ \ (\text{stiffness drops with damage}),\qquad w(0)=0,\quad w'(0)\ge0\ \ (\text{damage costs energy}). $$

Here $a(\alpha)$ is the **stiffness degradation function** (the note's $a$ is the code's `g`, e.g. $g(\alpha)=(1-\alpha)^2$ so $a'(0)=-2$), $w(\alpha)$ is the **local dissipation potential**, $G_c$ the fracture toughness, and $\ell_d$ the **damage length** (the note's $\ell_d$ — *not* the half-specimen length $\ell$). In the code $\ell_d$ is set by the third control parameter through $\ell_d=\hat\ell\,\ell$ (`l_hat` $\times$ `ell`); a larger $\hat\ell$ means a coarser, tougher damage band.

### 5.2 First-order nucleation condition

From the intact state $\alpha=0$, damage can only *grow* ($\dot\alpha\ge0$). It starts to grow when pushing the energy *downhill* in the $+\alpha$ direction is no longer penalised, i.e. when the first variation vanishes:

$$ \partial_\alpha\psi(e,0)=\tfrac12 E_h\,a'(0)\,e^2+\frac{G_c}{\ell_d}\,w'(0)=0. $$

Since $a'(0)<0$ the elastic term is *destabilising* (more strain $\Rightarrow$ stronger push to damage) while the fracture term is *stabilising*. Solving for the strain gives the **algebraic nucleation threshold** (PDF eq 21)

$$ \boxed{\,e^2\ \ge\ \varepsilon_{crit}^2:=\frac{2\,G_c\,w'(0)}{E_h\,\ell_d\,|a'(0)|}\,},\qquad N_{crit}=E_h\,\varepsilon_{crit}. $$

This is exactly [`critical_strain`](../tools/solvers.py), which gets $w'(0)$ and $a'(0)$ by a finite difference on the model lambdas. (Note the threshold depends only on $e^2$, so it is the same in either sign convention — the only thing the §1.2 sign fixes is the *sign* of the dynamic $e$ we compare against it, not the threshold.) **Key point for the third axis:** $\varepsilon_{crit}\propto1/\sqrt{\ell_d}=1/\sqrt{\hat\ell\,\ell}$, so at fixed geometry a *smaller* $\hat\ell$ raises $\varepsilon_{crit}$ (a finer damage band is harder to trigger) — this is how $\hat\ell$ tunes $\varepsilon_{crit}/\theta$.

### 5.3 AT1 vs AT2 — why the model choice matters ⚠️

The two standard phase-field models differ exactly in $w'(0)$:

| model | $w(\alpha)$ | $w'(0)$ | $\varepsilon_{crit}$ (with $G_c{=}E_h{=}\ell_d{=}1$, $a'(0){=}-2$) | sub-critical elastic phase? |
|---|---|---|---|---|
| **AT1** | $\alpha$ | $1$ | $\varepsilon_{crit}=\sqrt{G_c/(E_h\ell_d)}=1$ | **yes** — finite threshold |
| **AT2** | $\alpha^2$ | $0$ | $\varepsilon_{crit}=0$ | **no** — damages at any strain |

So **only AT1 gives a finite threshold** (the PDF's AT1 example: $a=(1-\alpha)^2$, $w=\alpha$, $\varepsilon_{crit}=\sqrt{G_c/(E_h\ell_d)}$). AT2 has $\varepsilon_{crit}=0$: damage grows from zero strain, so "did the wave pass a threshold?" makes no sense. **This study needs AT1** (the code cell below confirms both values). With $\theta=1$ and $\ell_d=1$ the AT1 threshold $\varepsilon_{crit}=1$ would sit the film exactly at the static threshold, so we push $\varepsilon_{crit}$ a little above $\theta$ — either by an explicit `e_crit` or, in the sweep of Section 7, by lowering $\hat\ell$ so that $\varepsilon_{crit}=1/\sqrt{\hat\ell\,\ell}>\theta$.

### 5.4 The secondary-cracking test

A second crack fires iff the **dynamic** strain reaches the threshold somewhere away from the already-relaxed first crack (PDF Criterion 1):

$$ |e(x,t)|\ \ge\ \varepsilon_{crit}\quad\text{for some }t>0\text{ and some }x\gtrsim\text{(a few }\ell_e\text{ from the crack)}. $$

The excluded zone (a few $\ell_e$ around $x=0$, where stress is already relaxed and the first damage band sits) is handled by [`effective_exclusion`](secondary.py)/[`detect_secondary_trigger`](secondary.py). The **trigger margin** is

$$ \mathcal R_{crack}=\max_{x,t}\frac{|e(x,t)|}{\varepsilon_{crit}},\qquad \mathcal R_{crack}\ge1\ \Longleftrightarrow\ \text{secondary crack},\qquad \mathcal R_{crack}=\mathcal R_{crack}(\Lambda,\Gamma,\hat\ell). $$

**The window that matters.** The film carries a pre-stress of magnitude $|e_0|=\theta$, so a film with $\varepsilon_{crit}<\theta$ would already have cracked statically — not interesting. The interesting range is

$$ 1<\frac{\varepsilon_{crit}}{\theta}\lesssim 2 : $$

the film is *just* below threshold statically, and the only way to break it again is a **dynamic overshoot** — the reflected, overlapping wave must briefly push $|e|$ above $\varepsilon_{crit}$ before damping kills it. Since $\varepsilon_{crit}/\theta=1/\sqrt{\hat\ell\,\ell}$ (with $\theta=1$, $\ell=\ell_e/\Lambda$), this window is exactly what the third control parameter $\hat\ell$ selects — small $\hat\ell$ pushes $\varepsilon_{crit}$ up into the interesting range.

In [10]:
print('e_crit (AT1):', critical_strain('AT1', Gc=1.0, E_h=1.0, ell_d=1.0))
print('e_crit (AT2):', critical_strain('AT2', Gc=1.0, E_h=1.0, ell_d=1.0), ' (no elastic phase)')

e_crit (AT1): 1.000000000263178
e_crit (AT2): 0.00010000000002631779  (no elastic phase)


## 6. Putting it together: one run

[`run_problem`](secondary.py) chains Steps 1–4 for one $(\Lambda,\Gamma,\hat\ell)$ triple: it builds the modes (§3), evolves them with damping (§4), rebuilds the space–time strain $e(x,t)$ (§4.4), and applies the trigger test (§5.4). The threshold is either derived from $\hat\ell$ (via $\ell_d=\hat\ell\,\ell$) or, as in the two runs below, pinned with an explicit `e_crit` so the geometry/damping contrast is clean. Then it calls `tools.plotting.plot_secondary_run` for a six-panel diagnostic that follows the note's list of questions in order:

1. **static release** — $\Delta E_{1/2}(\Lambda)$ (Step 1);
2. **modal energy quota** $Q_n$ vs $\beta_n$ (Step 2);
3. the **space–time strain map** $e(x,t)$ with the $|e|=e_{crit}$ contour on top;
4. the **overshoot envelope** $\max_t|e(x,t)|$ — *where* along the film a second crack would show up;
5. the **threshold-crossing history** $\max_x|e(x,t)|$ vs $t$, against the round-trips and the $e^{-\gamma t}$ decay envelope — *when* it would show up;
6. the **released-energy curve** for reference.

**Where and when the overshoot shows up.** It helps to know *what* the strain map shows before reading it: the biggest overshoot sits near the **far / symmetry end $x=\ell$** and appears **within the first half round-trip** ($t\sim\tfrac12\tau_{rt}$), not after many reflections. The reason: $x=\ell$ carries the condition $v(\ell,t)=0$ — a *strain antinode* where the released pulse piles up. So in 1D the "refocusing" is mostly the released shock arriving at the symmetry plane; later reflections only add to it. That is also why damping matters: it has only a fraction of a round-trip to act before that first pile-up.

The two runs below compare the two damping regimes of §4.2 at fixed geometry $\Lambda=\ell_e/\ell=0.2$ (i.e. $\ell=5\ell_e$, a long film) and a sub-critical threshold $e_{crit}=1.3>\theta=1$:

- **Weak damping** ($\Gamma=0.2$): the shock reaches the symmetry end almost undamped and the strain briefly **overshoots** $e_{crit}$ → second crack (`trigger` is not `None`).
- **Strong damping** ($\Gamma=3.0$): the pulse is weakened enough on the way to $x=\ell$ that the strain never reaches $e_{crit}$ → no second crack.

> **Why $e_{crit}=1.3$ and not $1.1$ here?** At $\Lambda=0.2$ ($\ell=5\ell_e$) the *undamped* overshoot is large, $\max|e|/\theta\approx2.2$ (run it: `trigger_map` margin at $\Gamma=0$). A threshold very close to $\theta$ (e.g. $1.1$) then cracks for *almost any* damping — even $\Gamma=3$ still overshoots — so it makes a poor illustration. Raising $e_{crit}$ to $1.3$ puts the weak/strong boundary at a moderate $\Gamma$, giving the clean contrast below. This sensitivity to $e_{crit}/\theta$ is exactly the third axis of the regime map ($\hat\ell$, Section 7).

Watch the printed `secondary crack` / `no secondary crack` lines and compare the panel-3 strain maps of the two runs.

In [ ]:
# Same geometry (Lambda=0.2, i.e. ell=5*ell_e) and threshold (e_crit=1.3 > theta=1), contrasting damping.
# Weak damping: the released shock overshoots e_crit -> secondary crack.
res_weak = sc.run_problem({'Lambda_bar': 0.2, 'Gamma': 0.2, 'e_crit': 1.3})

# Strong damping: the packet is killed before it overshoots -> no secondary crack.
res_strong = sc.run_problem({'Lambda_bar': 0.2, 'Gamma': 3.0, 'e_crit': 1.3})

print("\nweak  Gamma=0.2 :", "secondary crack" if res_weak['trigger'] else "no secondary crack")
print("strong Gamma=3.0 :", "secondary crack" if res_strong['trigger'] else "no secondary crack")

## 7. Regime maps — the criterion the note asks for

Steps 1–5 give a yes/no answer for one parameter set; the *phase diagram* comes from sweeping the control numbers. [`trigger_map`](secondary.py) runs `run_problem` over the $(\Lambda,\Gamma,\hat\ell)$ cube and records the **trigger margin**

$$ \mathcal R(\Lambda,\Gamma,\hat\ell)=\max_{x,t}\frac{|e(x,t)|}{e_{crit}},\qquad e_{crit}=\frac{1}{\sqrt{\hat\ell\,\ell}}\ \ (\text{AT1},\,G_c{=}E_h{=}1). $$

For **each** value of $\hat\ell$ it sweeps the $(\Lambda,\Gamma)$ plane and draws a heat-map (via `tools.plotting.plot_secondary_regime_map`), whose **level-1 contour $\mathcal R=1$ is the secondary-cracking boundary**:

- $\mathcal R<1$ — **no second crack** (smooth relaxation; the wave never reaches threshold again);
- $\mathcal R\approx1$ — **near threshold** (a second crack is marginal, sensitive to parameters);
- $\mathcal R>1$ — **likely second crack** (and, repeating, a possible fragmentation cascade).

This is exactly the short answer the note wants: one inequality $\mathcal R(\Lambda,\Gamma,\hat\ell)\ge1$ organising the change from a single crack to fragmentation. Reading a map, the boundary moves the intuitive way — **more damping** (larger $\Gamma$) suppresses second cracks, while **geometry** $\Lambda=\ell_e/\ell$ sets both how much energy is released (Step 1) and how the round-trip time scales (Step 4); the **damage length** $\hat\ell$ shifts the whole boundary through $e_{crit}\propto1/\sqrt{\hat\ell}$.

The sweep below produces **ten figures**: one $(\Lambda,\Gamma)$ regime map for **each of the nine $\hat\ell$ values** (file name carries $\hat\ell$ so they do not overwrite), plus a single **paper-style summary** (`plot_secondary_regime_summary`) that overlays every $\mathcal R=1$ boundary in one $(\Lambda,\Gamma)$ plane, coloured by $\hat\ell$. Note we do **not** pass an explicit `e_crit` here — letting $\hat\ell$ set the threshold is the whole point of the third axis. It uses the coarse default resolution (`n_modes=120, n_x=200, n_t=400`) that `trigger_map` sets internally for speed; nine full planes still take a little while.

In [ ]:
# Sweep the (Lambda, Gamma) plane for nine damage-length ratios l_hat.
# Lambda = ell_e/ell in [0.1, 2] spans long films (small Lambda) to short ones;
# e_crit is derived from l_hat (via ell_d = l_hat*ell), so DON'T pin e_crit here.
# -> nine (Lambda, Gamma) regime maps (one per l_hat) + one summary of R=1 boundaries.
sweep = sc.trigger_map(Lambda_bars=np.linspace(0.1, 2.0, 12),
                       Gammas=np.linspace(0.0, 2.0, 9),
                       l_hats=np.geomspace(0.02, 0.5, 9))

## 8. Summary and open questions

### What we built

A fully semi-analytic chain answering the note's main questions:

| # | question | where | answer |
|---|---|---|---|
| 1 | How much energy does a first crack release? | §2 | $\Delta E_{1/2}=\tfrac12 E_h\theta^2\ell_e\tanh(\ell/\ell_e)=\tfrac12 E_h\theta^2\ell_e\tanh(1/\Lambda)$ — levels off at $\tfrac12 E_h\theta^2\ell_e$ for long films ($\Lambda\ll1$) |
| 2–3 | Which wavelengths carry it / how does $\ell_e$ filter? | §3 | exact Lorentzian $a_n\propto1/(1+\beta_n^2)$; energy peaks at $\beta_n\sim1$ |
| 4 | How much damping suppresses second cracks? | §4 | the round-trip damping number $\Gamma=\gamma\tau_{rt}$; suppression for $\Gamma\gtrsim1$ |
| 5 | Can reflected waves refocus enough to nucleate? | §4–6 | yes, when the dynamic overshoot pushes $|e|$ past $e_{crit}$ before $e^{-\gamma t}$ kills it |
| 6 | Which parameters organise the change? | §7 | the single inequality $\mathcal R(\Lambda,\Gamma,\hat\ell)\ge1$, with $\Lambda=\ell_e/\ell$, $\Gamma=\gamma\tau_{rt}$, $\hat\ell=\ell_d/\ell$ (the last sets $e_{crit}\propto1/\sqrt{\hat\ell}$) |

### Open questions / next steps

- **Check against the FEM.** Compare this threshold map with the full phase-field dynamic runs ([`problems/dynamic.py`](dynamic.py)): do the FEM second-crack events fall on the predicted side of the $\mathcal R=1$ boundary? (See the note↔code map and the FEM secondary experiment in the project notes.)
- **The test is local in strain.** The model also pays the **gradient term** $G_c\ell_d|\alpha_x|^2$ — making a real crack means building a damage band of width $\sim\ell_d$, which costs more than the pointwise threshold. So the semi-analytic boundary should be a little **optimistic** (predict second cracks slightly too easily); measuring that gap is a natural refinement.
- **Bound the useful window.** Find the dynamic overshoot factor $\max|e|/\theta$ as a function of $\Lambda$ alone in the **undamped** limit ($\Gamma\to0$) — it caps the useful range of $e_{crit}/\theta$ (i.e. of $\hat\ell$) where a second crack is ever possible.
- **Fragmentation cascade.** Once a second crack forms, the domain splits into new sub-specimens, each with its own $\Lambda$ and $\hat\ell$; repeating the analysis turns the single rule into a semi-analytic **fragmentation cascade**.